# Personal finance cleaning notebook

This notebook demonstrates a public-safe cleaning pipeline for a fictional household finance export.

Key idea: keep the raw sample messy, then make every cleaning and classification decision reproducible in code or in `data/reference/classification_rules.csv`.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.finance_pipeline import (
    load_raw_table,
    prepare_transaction_area,
    reshape_transactions,
    load_rules,
    apply_rules,
    finalize_dataset,
    build_quality_outputs,
)

sns.set_theme(style="whitegrid")

PROJECT_DIR = Path.cwd()
RAW_FILE = PROJECT_DIR / "data" / "sample" / "raw_transactions_sample.csv"
RULES_FILE = PROJECT_DIR / "data" / "reference" / "classification_rules.csv"
CLEAN_OUTPUT = PROJECT_DIR / "data" / "sample" / "cleaned_transactions_sample.csv"
SUMMARY_OUTPUT = PROJECT_DIR / "data" / "quality" / "cleaning_summary_sample.csv"
UNMAPPED_OUTPUT = PROJECT_DIR / "data" / "quality" / "unmapped_details_sample.csv"

for path in [CLEAN_OUTPUT.parent, SUMMARY_OUTPUT.parent, UNMAPPED_OUTPUT.parent]:
    path.mkdir(parents=True, exist_ok=True)

RAW_FILE, RULES_FILE

## 2. Load and inspect the messy source export

In [ ]:
raw = load_raw_table(RAW_FILE)
print(f"Raw shape: {raw.shape}")
display(raw.head(12))

## 3. Keep the transaction area and forward-fill dates

The sample file imitates a wide personal finance tracker:
- one row can hold multiple category amounts
- continuation rows can omit the date
- header rows are noisy and should be excluded from the transaction area

In [ ]:
transaction_area = prepare_transaction_area(raw)
print(f"Transaction area rows: {len(transaction_area):,}")
display(transaction_area.head(12))

## 4. Reshape from wide layout to one transaction per row

In [ ]:
tidy = reshape_transactions(transaction_area)
print(f"Tidy transactions: {len(tidy):,}")
display(tidy.head(20))

## 5. Apply rule-based categorization

The pipeline uses two layers:
1. exact rules from `classification_rules.csv`
2. keyword fallbacks for obvious new variants

Anything still unresolved is marked `needs_review` and exported to a separate quality file.

In [ ]:
rules = load_rules(RULES_FILE)
cleaned = apply_rules(tidy, rules)
cleaned = finalize_dataset(cleaned, RAW_FILE)
print(f"Rules loaded: {len(rules):,}")
display(cleaned.head(20))

## 6. Run quality checks and export reproducible outputs

In [ ]:
summary, unmapped = build_quality_outputs(cleaned, rules)
cleaned.to_csv(CLEAN_OUTPUT, index=False)
summary.to_csv(SUMMARY_OUTPUT, index=False)
unmapped.to_csv(UNMAPPED_OUTPUT, index=False)

display(summary)
display(unmapped.head(15))

print(CLEAN_OUTPUT)
print(SUMMARY_OUTPUT)
print(UNMAPPED_OUTPUT)

## 7. Quick validation charts

In [ ]:
monthly = (
    cleaned.groupby(["year_month", "transaction_direction"], as_index=False)["amount"].sum()
    .pivot(index="year_month", columns="transaction_direction", values="amount")
    .fillna(0)
    .reset_index()
)
monthly["savings_rate"] = (monthly["income"] - monthly["expense"]) / monthly["income"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(monthly["year_month"], monthly["income"], marker="o", label="Income")
axes[0].plot(monthly["year_month"], monthly["expense"], marker="o", label="Expenses")
axes[0].set_title("Income vs Expenses")
axes[0].tick_params(axis="x", rotation=35)
axes[0].legend(frameon=False)

category_spend = cleaned.loc[cleaned["transaction_direction"].eq("expense")].groupby("category", as_index=False)["amount"].sum()
sns.barplot(data=category_spend, x="amount", y="category", ax=axes[1], color="#4c78a8")
axes[1].set_title("Expense categories")
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("")
plt.tight_layout()

## 8. Why this public sample exists

The original private project used real household transactions. Those rows are intentionally excluded from the public repository.

This sample keeps the useful engineering problems:
- inconsistent merchant text
- duplicate merchant variants
- blank continuation dates
- malformed numeric formatting
- unresolved descriptions that need analyst review

That makes the repo safe to publish without pretending the messy part never existed.